# 🚀 Multi-Modal RAG System (End-to-End Resume Portfolio Project)
### Complete Retrieval-Augmented Generation for Text, PDF, Images, and Video
**Data Sources:** Web Scraping, Excel Spreadsheets, PDF Documents, Datasets (JSON), Images, and Timestamped Videos  
**Tech Stack:** LangChain, ChromaDB Vector Store, HuggingFace Embeddings, OpenCV, PyMuPDF/pypdf, Pandas, Google Gemini / Groq LLM

---

### 🧠 System Architecture Overview
```
┌────────────────────────────────────────────────────────────────────────┐
│                        1. MULTI-MODAL INGESTION                        │
│   🌐 Web Scraper  │  📊 Excel (.xlsx)  │  📑 PDF Documents             │
│   📁 Datasets     │  🖼️ Images (PIL)   │  🎥 Video (OpenCV Keyframes)  │
└──────────────────────────────────┬─────────────────────────────────────┘
                                   │
                                   ▼
┌────────────────────────────────────────────────────────────────────────┐
│                  2. SMART MULTI-MODAL CHUNKING                         │
│   • Text Splitter for Web Content                                      │
│   • Page-by-Page Sliding Window Chunks for PDF Documents               │
│   • Schema-Aware Markdown Rows for Spreadsheets                        │
│   • Timestamped Segments (MM:SS) & Frame Paths for Videos              │
└──────────────────────────────────┬─────────────────────────────────────┘
                                   │
                                   ▼
┌────────────────────────────────────────────────────────────────────────┐
│                 3. VECTOR EMBEDDINGS (Local & Fast)                    │
│   • HuggingFace / Sentence-Transformers (all-MiniLM-L6-v2)             │
│   • Zero rate limits, runs directly on Colab CPU / GPU                 │
└──────────────────────────────────┬─────────────────────────────────────┘
                                   │
                                   ▼
┌────────────────────────────────────────────────────────────────────────┐
│                  4. CHROMADB MULTI-MODAL VECTOR STORE                  │
│   • Embedded SQLite Persistent Storage                                 │
│   • Stores text vectors + cross-modal image, PDF & video metadata      │
└──────────────────────────────────┬─────────────────────────────────────┘
                                   │
                                   ▼
┌────────────────────────────────────────────────────────────────────────┐
│                   5. RETRIEVER & CROSS-MODAL SEARCH                    │
│   • Top-k Semantic Similarity Search                                   │
│   • Fetches relevant rows, PDF pages, text chunks, and video frames    │
└──────────────────────────────────┬─────────────────────────────────────┘
                                   │
                                   ▼
┌────────────────────────────────────────────────────────────────────────┐
│                   6. MULTIMODAL LLM REASONING & QA                     │
│   • Synthesizes answers citing exact rows, PDF pages and timestamps    │
│   • Displays attached keyframe images and diagrams inline              │
└────────────────────────────────────────────────────────────────────────┘
```

## 🛠️ Step 1: Install Dependencies

In [ ]:
# Install core RAG, vector store, and multimodal dependencies
!pip install -q -U langchain langchain-community langchain-chroma chromadb sentence-transformers
!pip install -q -U pandas openpyxl pillow opencv-python-headless beautifulsoup4 requests tabulate pypdf pymupdf
!pip install -q -U google-generativeai langchain-google-genai langchain-groq
print("✅ All dependencies installed successfully!")

## 🔑 Step 2: Configure API Keys
You can use either **Google Gemini API** or **Groq API** (or both!) for generating answers.  
*(Tip: Get a free key at [Google AI Studio](https://aistudio.google.com/) or [Groq Console](https://console.groq.com/))*

In [ ]:
import os

# Check .env file if running in VS Code / local environment
if os.path.exists(".env"):
    with open(".env", "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                k, v = line.split("=", 1)
                k, v = k.strip(), v.strip().strip("'\"")
                if k and not os.environ.get(k):
                    os.environ[k] = v

# Check Colab secrets or prompt user
try:
    from google.colab import userdata
    gemini_key = userdata.get('GEMINI_API_KEY')
    if gemini_key:
        os.environ["GEMINI_API_KEY"] = gemini_key
        print("✅ Loaded GEMINI_API_KEY from Colab secrets.")
except Exception:
    pass

if "GEMINI_API_KEY" not in os.environ:
    key = input("Enter your GEMINI_API_KEY (or press Enter if using GROQ): ").strip()
    if key:
        os.environ["GEMINI_API_KEY"] = key

# Optional: Groq API Key
if "GROQ_API_KEY" not in os.environ:
    groq_key = input("Optional: Enter your GROQ_API_KEY (or press Enter to skip): ").strip()
    if groq_key:
        os.environ["GROQ_API_KEY"] = groq_key

print("✅ API Key configuration ready.")

## 📥 Step 3: Multi-Modal Data Ingestion (6 Modalities: Web, Excel, PDF, JSON, Image, Video)
In a production enterprise RAG system, data comes in diverse formats.  
Here we ingest **Web pages**, **Excel files**, **PDF Documents**, **Structured JSON**, **Images**, and **Videos**.

In [ ]:
import os
import json
import cv2
import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
from PIL import Image, ImageDraw

data_dir = "./multimodal_raw_data"
os.makedirs(data_dir, exist_ok=True)

# ---------------------------------------------------------
# 1. Web Scraping Ingestion
# ---------------------------------------------------------
web_url = "https://en.wikipedia.org/wiki/Retrieval-augmented_generation"
resp = requests.get(web_url, headers={"User-Agent": "Mozilla/5.0"})
soup = BeautifulSoup(resp.text, "html.parser")
for tag in soup(["script", "style", "nav", "footer"]):
    tag.extract()
raw_web_text = "\n".join([p.get_text() for p in soup.find_all("p") if len(p.get_text().strip()) > 40])
print(f"🌐 Web Page Scraped: {len(raw_web_text)} characters from {web_url}")

# ---------------------------------------------------------
# 2. Excel Spreadsheet Ingestion
# ---------------------------------------------------------
excel_path = os.path.join(data_dir, "hardware_sales_q3.xlsx")
df_sales = pd.DataFrame({
    "Product_ID": ["HW-101", "HW-102", "HW-103", "HW-104"],
    "Product_Name": ["H100 Tensor GPU", "Quantum Accelerator Q1", "Edge Vision Module", "Autonomous Robot Controller"],
    "Category": ["AI Compute", "Quantum Systems", "Edge AI", "Robotics"],
    "Q2_Revenue_USD": [3200000, 1100000, 750000, 920000],
    "Q3_Revenue_USD": [4500000, 1400000, 890000, 1150000],
    "Growth_Percent": ["40.6%", "27.2%", "18.6%", "25.0%"],
    "Region": ["Global", "North America", "Asia-Pacific", "Europe"]
})
with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    df_sales.to_excel(writer, sheet_name="Q3_Sales_Report", index=False)
print(f"📊 Excel Sheet Created: {excel_path} ({len(df_sales)} rows)")

# ---------------------------------------------------------
# 3. Dataset Ingestion (JSON)
# ---------------------------------------------------------
dataset_path = os.path.join(data_dir, "ai_safety_benchmarks.json")
benchmark_records = [
    {"benchmark_id": "MMLU-PRO-01", "test_domain": "Computer Science", "score": 91.4, "baseline": 85.0, "status": "SUPERIOR"},
    {"benchmark_id": "MATH-HARD-02", "test_domain": "Complex Analysis", "score": 88.7, "baseline": 79.2, "status": "SUPERIOR"},
    {"benchmark_id": "VIDEO-QA-03", "test_domain": "Video Question Answering", "score": 95.2, "baseline": 82.1, "status": "STATE_OF_THE_ART"}
]
with open(dataset_path, "w", encoding="utf-8") as f:
    json.dump(benchmark_records, f, indent=2)
print(f"📁 Dataset Created: {dataset_path} ({len(benchmark_records)} records)")

# ---------------------------------------------------------
# 4. Image Ingestion (Diagram)
# ---------------------------------------------------------
image_path = os.path.join(data_dir, "multimodal_architecture_diagram.png")
img = Image.new("RGB", (640, 360), color=(24, 28, 40))
draw = ImageDraw.Draw(img)
# Draw boxes
draw.rectangle([40, 60, 220, 160], outline=(66, 153, 225), width=3)
draw.text((60, 100), "Multi-Source Ingestion\n(Web, Excel, Video)", fill=(255, 255, 255))
draw.rectangle([420, 60, 600, 160], outline=(72, 187, 120), width=3)
draw.text((440, 100), "ChromaDB Vector Store\n(Unified Embeddings)", fill=(255, 255, 255))
draw.rectangle([230, 210, 410, 310], outline=(237, 137, 54), width=3)
draw.text((250, 250), "Multimodal LLM\nSynthesis Engine", fill=(255, 255, 255))
img.save(image_path)
print(f"🖼️ Image Diagram Created: {image_path}")

# ---------------------------------------------------------
# 5. Video Ingestion with OpenCV
# ---------------------------------------------------------
video_path = os.path.join(data_dir, "drone_surveillance_demo.mp4")
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
fps, duration, width, height = 10, 6, 480, 270
writer = cv2.VideoWriter(video_path, fourcc, fps, (width, height))
scenes = [
    ("Phase 1: Drone Liftoff & GPS Lock", (20, 20, 80)),
    ("Phase 2: Thermal Sensor Calibration", (80, 20, 20)),
    ("Phase 3: Structural Inspection Complete", (20, 80, 20))
]
for i in range(fps * duration):
    scene_idx = min(i // (fps * 2), len(scenes) - 1)
    caption, color = scenes[scene_idx]
    frame = np.full((height, width, 3), color, dtype=np.uint8)
    cv2.putText(frame, f"TIMESTAMP: {i/fps:.1f}s", (25, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
    cv2.putText(frame, caption, (25, 140), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 255, 200), 2)
    writer.write(frame)
writer.release()
print(f"🎥 Video Created: {video_path} (Duration: 6s, 3 distinct visual phases)")
# ---------------------------------------------------------
# 2.5 PDF Document Ingestion
# ---------------------------------------------------------
pdf_path = os.path.join(data_dir, 'ai_architecture_spec.pdf')
try:
    import fitz
    pdf_doc = fitz.open()
    p1 = pdf_doc.new_page()
    p1.insert_text((50, 70), 'Artificial Intelligence Safety & Multi-Modal Architecture Guidelines\n\nSection 1: Multi-Modal Ingestion Protocols\nThe system ingests text, spreadsheets, datasets, images, videos, and PDFs.\nFor high-concurrency workloads, chunk sizes should be capped at 600 words with 100 word overlap.\n\nSection 2: Security & Privacy Compliance\nAll API keys and credentials must be stored securely in environment variables.\nNever log sensitive user query vectors directly to unencrypted persistent stores.\n\nSection 3: Model Architecture Recommendations\nUse Gemini-1.5-Flash or Gemini-2.5-Flash for ultra-low latency response synthesis.')
    pdf_doc.save(pdf_path)
    pdf_doc.close()
    print(f'📑 PDF Document Created: {pdf_path}')
except Exception:
    pass


### 👁️ Previewing the Ingested Media Assets
Let's display the generated architecture image directly in Colab:

In [ ]:
from IPython.display import display

# Display the diagram
display(Image.open(image_path))

# Display the Excel Table preview
print("\n--- Excel Spreadsheet Preview ---")
display(pd.read_excel(excel_path))

## ✂️ Step 4: Multi-Modal Chunking & Semantic Representation
> **Interview Concept:** In a standard RAG system, chunking is just `RecursiveCharacterTextSplitter` on text.  
> But in a **Multi-Modal RAG** system, chunking must be **modality-aware**:
- **Web Text:** Chunked by token/character length with sliding overlap.
- **Excel/Tables:** Chunked by markdown table blocks AND natural row-level sentences (`Product: X | Revenue: $Y`).
- **Images:** Chunked by visual layout description, OCR text, and resolution metadata.
- **Videos:** Chunked by **time-based keyframe sampling** using OpenCV, extracting exact timestamps (`MM:SS`) and frame file paths!

In [ ]:
from langchain.schema import Document
from langchain_text_splitters import CharacterTextSplitter

all_chunks = []

# ---------------------------------------------------------
# 4.1 Chunking Web Text
# ---------------------------------------------------------
text_splitter = CharacterTextSplitter(chunk_size=600, chunk_overlap=100)
web_splits = text_splitter.split_text(raw_web_text)
for i, chunk in enumerate(web_splits):
    all_chunks.append(Document(
        page_content=f"[Web Article: Retrieval-Augmented Generation]\n{chunk}",
        metadata={"source": web_url, "modality": "web_text", "chunk_id": i}
    ))
print(f"✅ Web Text Chunks created: {len(web_splits)}")

# ---------------------------------------------------------
# 4.2 Chunking Excel Spreadsheets (Markdown + Semantic Rows)
# ---------------------------------------------------------
df_excel = pd.read_excel(excel_path)
# Summary chunk
summary_content = (
    f"[Spreadsheet: hardware_sales_q3.xlsx | Sheet: Q3_Sales_Report]\n"
    f"Columns: {', '.join(df_excel.columns)}\nTotal Rows: {len(df_excel)}\n"
    f"Markdown Table:\n{df_excel.to_markdown(index=False)}"
)
all_chunks.append(Document(
    page_content=summary_content,
    metadata={"source": "hardware_sales_q3.xlsx", "modality": "table_summary"}
))

# Row-by-row natural language chunks for precise semantic matching
for idx, row in df_excel.iterrows():
    row_str = " | ".join([f"{col}: {val}" for col, val in row.items()])
    all_chunks.append(Document(
        page_content=f"[Spreadsheet Record: Row {idx+1}] {row_str}",
        metadata={"source": "hardware_sales_q3.xlsx", "modality": "table_row", "row": idx+1}
    ))
print(f"✅ Excel Chunks created: 1 summary + {len(df_excel)} row chunks")

# ---------------------------------------------------------
# 4.3 Chunking Dataset Records (JSON)
# ---------------------------------------------------------
for idx, record in enumerate(benchmark_records):
    all_chunks.append(Document(
        page_content=f"[Dataset Record #{idx+1}] Benchmark: {record['benchmark_id']} | Domain: {record['test_domain']} | Score: {record['score']} | Baseline: {record['baseline']} | Status: {record['status']}",
        metadata={"source": "ai_safety_benchmarks.json", "modality": "dataset_record", "id": record['benchmark_id']}
    ))
print(f"✅ Dataset Chunks created: {len(benchmark_records)}")

# ---------------------------------------------------------
# 4.4 Chunking Image (Diagram)
# ---------------------------------------------------------
image_content = (
    "[Image: multimodal_architecture_diagram.png]\n"
    "System Architecture Overview:\n"
    "1. Turquoise Box: 'Multi-Source Ingestion (Web, Excel, Video)' on the top-left.\n"
    "2. Green Box: 'ChromaDB Vector Store (Unified Embeddings)' on the top-right.\n"
    "3. Orange Box: 'Multimodal LLM Synthesis Engine' centered at the bottom.\n"
    "Resolution: 640x360 | Theme: Dark mode with color-coded modular pipelines."
)
all_chunks.append(Document(
    page_content=image_content,
    metadata={"source": "multimodal_architecture_diagram.png", "modality": "image", "file_path": image_path}
))
print("✅ Image Chunk created: 1 visual description chunk")

# ---------------------------------------------------------
# 4.5 Chunking Video (Keyframes at Timestamps via OpenCV)
# ---------------------------------------------------------
frames_dir = "./extracted_video_keyframes"
os.makedirs(frames_dir, exist_ok=True)

cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS) or 10.0
video_chunks_count = 0

frame_interval_sec = 2.0  # Sample every 2 seconds
frame_step = int(fps * frame_interval_sec)

scene_descriptions = [
    "Scene 1: Drone Liftoff & GPS Lock. Visual shows dark blue background with flight telemetry status.",
    "Scene 2: Thermal Sensor Calibration. Visual shows dark red background with thermal calibration progress.",
    "Scene 3: Structural Inspection Complete. Visual shows dark green background with all inspection targets passed."
]

frame_idx = 0
while cap.isOpened():
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ret, frame = cap.read()
    if not ret:
        break
    
    sec = frame_idx / fps
    timestamp_str = f"{int(sec // 60):02d}:{int(sec % 60):02d}"
    frame_file = os.path.join(frames_dir, f"drone_demo_t{int(sec):04d}.jpg")
    cv2.imwrite(frame_file, frame)
    
    desc_idx = min(int(sec // 2), len(scene_descriptions) - 1)
    desc = scene_descriptions[desc_idx]
    
    video_content = (
        f"[Video: drone_surveillance_demo.mp4 | Timestamp: {timestamp_str}]\n"
        f"Visual Scene: {desc}\n"
        f"Keyframe Image Path: {frame_file}"
    )
    
    all_chunks.append(Document(
        page_content=video_content,
        metadata={
            "source": "drone_surveillance_demo.mp4",
            "modality": "video_keyframe",
            "timestamp": timestamp_str,
            "frame_path": os.path.abspath(frame_file)
        }
    ))
    video_chunks_count += 1
    frame_idx += frame_step

cap.release()
print(f"✅ Video Chunks created: {video_chunks_count} timestamped keyframes")
print(f"\n🎉 TOTAL MULTI-MODAL CHUNKS PRODUCED: {len(all_chunks)}")
# ---------------------------------------------------------
# 4.6 Chunking PDF Documents (Page-by-page + Sliding Window)
# ---------------------------------------------------------
if os.path.exists(pdf_path):
    pdf_texts = []
    try:
        import fitz
        pdoc = fitz.open(pdf_path)
        pdf_texts = [page.get_text() for page in pdoc]
    except Exception:
        try:
            import pypdf
            reader = pypdf.PdfReader(pdf_path)
            pdf_texts = [p.extract_text() or '' for p in reader.pages]
        except Exception:
            pass
    for p_num, p_text in enumerate(pdf_texts, start=1):
        if p_text.strip():
            words = p_text.split()
            chunk_size, overlap = 400, 80
            for start_idx in range(0, len(words), chunk_size - overlap):
                sub_chunk = ' '.join(words[start_idx : start_idx + chunk_size])
                all_chunks.append(Document(
                    page_content=f'[Document: ai_architecture_spec.pdf | Page {p_num}]\n{sub_chunk}',
                    metadata={'source': 'ai_architecture_spec.pdf', 'modality': 'pdf_text', 'page': p_num}
                ))
    print(f'✅ PDF Chunks created from ai_architecture_spec.pdf')


### 🔍 Inspecting the Chunks (Recruiter Walkthrough)
Let's look at how chunks from different modalities look inside memory:

In [ ]:
print("=== Chunk #1: Web Chunk ===")
print(all_chunks[0].page_content[:250], "...\n")

print("=== Chunk #2: Excel Row Chunk ===")
for c in all_chunks:
    if c.metadata.get("modality") == "table_row":
        print(c.page_content)
        break

print("\n=== Chunk #3: Video Keyframe Chunk ===")
for c in all_chunks:
    if c.metadata.get("modality") == "video_keyframe":
        print(c.page_content)
        print("Metadata:", c.metadata)
        break

## 🧬 Step 5: Vector Embeddings (Local & Fast)
> **Why this matters for your Resume:**  
> Using **HuggingFace Sentence-Transformers (`all-MiniLM-L6-v2`)** runs locally on Google Colab's CPU/GPU.  
> Unlike API-based embeddings (like OpenAI/Google), local embeddings have **ZERO rate limits**, **ZERO cost**, and will **NEVER throw a 429 or 404 error**!

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

# Load lightweight, high-performance semantic embedding model
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Test embedding dimension
sample_vector = embeddings.embed_query("Quantum Accelerator Q1")
print(f"✅ Embedding Model Loaded: all-MiniLM-L6-v2")
print(f"Vector Dimension: {len(sample_vector)} (Float32 values per chunk)")

## 🗄️ Step 6: ChromaDB Multi-Modal Vector Database
We now insert all `all_chunks` into **ChromaDB**.  
Chroma builds an approximate nearest neighbor (ANN) HNSW index over our 384-dimensional vectors.

In [ ]:
from langchain_chroma import Chroma

persist_dir = "./chroma_multimodal_db"

# Create and populate Chroma Vector Database from our multimodal documents
vectordb = Chroma.from_documents(
    documents=all_chunks,
    embedding=embeddings,
    persist_directory=persist_dir
)

# Wrap as standard LangChain retriever
retriever = vectordb.as_retriever(search_kwargs={"k": 3})

print(f"✅ ChromaDB Initialized & Persisted at: {persist_dir}")
print(f"Total Documents Indexed: {len(all_chunks)}")

## 🎯 Step 7: Retriever in Action (Cross-Modal Similarity Search)
Before even calling the LLM, let's test the **Retriever directly**.  
Notice how querying for a video concept retrieves the exact keyframe and timestamp!

In [ ]:
# Query 1: Search the Spreadsheet
query_excel = "Which product has 40.6% growth in the sales report?"
retrieved_excel = retriever.invoke(query_excel)
print("🔍 QUERY:", query_excel)
print("📄 RETRIEVED CHUNK:", retrieved_excel[0].page_content)
print("-" * 60)

# Query 2: Search the Video by Visual Action
query_video = "Where does thermal sensor calibration happen in the drone video?"
retrieved_video = retriever.invoke(query_video)
print("🔍 QUERY:", query_video)
print("🎥 RETRIEVED CHUNK:", retrieved_video[0].page_content)
print("⏰ Timestamp:", retrieved_video[0].metadata.get("timestamp"))

# Display the retrieved video keyframe!
frame_path = retrieved_video[0].metadata.get("frame_path")
if frame_path and os.path.exists(frame_path):
    print(f"🖼️ Rendering Keyframe ({frame_path}):")
    display(Image.open(frame_path))

## 🤖 Step 8: Multi-Modal LLM Synthesis & QA Chain
We now connect our **ChromaDB Retriever** to a **Generative LLM**.  
You can run it with **Google Gemini** or **Groq LLaMA-3.3-70B**.

In [ ]:
# Initialize LLM (supports Google Gemini or Groq LLaMA-3)
llm = None

# 1. Try Google Gemini
if os.environ.get("GEMINI_API_KEY"):
    try:
        import google.generativeai as genai
        genai.configure(api_key=os.environ["GEMINI_API_KEY"])
        # Use standard Google Generative AI model
        class GeminiLLMWrapper:
            def __init__(self, model_name="gemini-1.5-flash"):
                self.model = genai.GenerativeModel(model_name)
            def invoke(self, prompt_text):
                resp = self.model.generate_content(prompt_text)
                return resp.text.strip()
        llm = GeminiLLMWrapper("gemini-1.5-flash")
        print("✅ Gemini 1.5 Flash initialized as QA Generator.")
    except Exception as e:
        print(f"Gemini init note: {e}")

# 2. Fallback to Groq LLaMA-3 if available
if llm is None and os.environ.get("GROQ_API_KEY"):
    try:
        from langchain_groq import ChatGroq
        groq_model = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
        class GroqLLMWrapper:
            def __init__(self, m):
                self.m = m
            def invoke(self, prompt_text):
                return self.m.invoke(prompt_text).content
        llm = GroqLLMWrapper(groq_model)
        print("✅ Groq LLaMA-3.3-70B initialized as QA Generator.")
    except Exception as e:
        print(f"Groq init note: {e}")

if llm is None:
    print("⚠️ No API key active. Retrieval mode will display retrieved chunks and visual evidence.")

### 🎯 Step 9: Running Multi-Modal RAG Queries
The function below implements the full RAG pipeline:
1. Queries ChromaDB for top-3 chunks.
2. Formats the retrieved multimodal context.
3. Generates the synthesized answer with citations.
4. Displays any attached images or video keyframes inline!

In [ ]:
from IPython.display import display, Markdown
import time

def ask_multimodal_rag(question: str):
    print("=" * 75)
    display(Markdown(f"### ❓ **Question**: {question}"))
    
    # 1. Retrieve relevant chunks
    docs = retriever.invoke(question)
    
    # 2. Gather text context and visual assets
    context_parts = []
    visual_assets = []
    
    for i, d in enumerate(docs, 1):
        context_parts.append(f"[Source #{i} | {d.metadata.get('modality', 'doc')}]:\n{d.page_content}")
        
        # Check if this retrieved document points to an image or video keyframe
        img_file = d.metadata.get("file_path") or d.metadata.get("frame_path")
        if img_file and os.path.exists(img_file) and img_file not in visual_assets:
            visual_assets.append(img_file)
            
    context_text = "\n\n".join(context_parts)
    
    # 3. Formulate Prompt
    prompt = (
        "You are an expert Multi-Modal RAG system. Answer the question based on the retrieved context below.\n"
        "- If citing spreadsheets, cite exact rows, columns, and product figures.\n"
        "- If citing videos, cite the exact timestamp (e.g. 'At timestamp 00:02 in the drone video...').\n"
        "- If citing images/diagrams, describe the relevant blocks and visual components.\n\n"
        f"=== RETRIEVED MULTIMODAL CONTEXT ===\n{context_text}\n\n"
        f"=== USER QUESTION ===\n{question}\n\n"
        "=== DETAILED ANSWER ==="
    )
    
    # 4. Generate Answer via LLM
    if llm:
        try:
            answer = llm.invoke(prompt)
        except Exception as e:
            answer = f"[API Cooldown - Showing Retrieved Evidence below]\n{e}"
    else:
        answer = f"[Retrieval Mode]\nMost relevant context:\n{docs[0].page_content}"
        
    display(Markdown(f"### 🤖 **Answer**:\n{answer}"))
    
    # 5. Display Visual Evidence Inline
    if visual_assets:
        print("\n🖼️ Attached Visual Evidence / Video Keyframes:")
        for path in visual_assets:
            print(f"- {path}")
            try:
                display(Image.open(path))
            except Exception:
                pass

# ---------------------------------------------------------
# Run Test Queries across All Modalities
# ---------------------------------------------------------

# Query 1: Tabular Excel Question
ask_multimodal_rag("Which product had the highest Q3 revenue in the sales report, and what was its growth percent?")
time.sleep(3)

# Query 2: Video Timestamp Question
ask_multimodal_rag("What happens at timestamp 00:02 in the drone inspection video?")
time.sleep(3)

# Query 3: Architecture Diagram Question
ask_multimodal_rag("What blocks and components are shown in the system architecture diagram?")
time.sleep(3)

# Query 4: Web Scraped Question
ask_multimodal_rag("According to the scraped Wikipedia article, what is Retrieval-Augmented Generation?")
# Query 4: PDF Guidelines Question
ask_multimodal_rag('What are the recommended chunk sizes and safety protocols in the AI guidelines PDF?')
time.sleep(3)


## 🎓 Resume & Recruiter Summary Cheat-Sheet

Use these key bullet points to describe this project on your **Resume / LinkedIn / Interview**:

### 📌 Resume Project Description:
**Title: End-to-End Multi-Modal Retrieval-Augmented Generation (RAG) System**
- **Engineered a Multi-Modal RAG pipeline** supporting heterogeneous data sources including scraped web articles, multi-sheet Excel spreadsheets, JSON datasets, system diagrams, and MP4 video files.
- **Built modality-aware chunking algorithms**: created semantic markdown-table row representations for tabular data and implemented an **OpenCV keyframe extraction pipeline** to sample video frames with exact `MM:SS` timestamp metadata.
- **Indexed multimodal embeddings** using `sentence-transformers/all-MiniLM-L6-v2` within a **ChromaDB** vector database, achieving fast cosine similarity retrieval across modalities.
- **Integrated LangChain and Generative LLMs** to synthesize grounded answers with verifiable citations (exact spreadsheet rows and video timestamps) and render inline visual evidence.

### 💡 Key Questions Recruiters Will Ask You:
1. **"How did you chunk non-text data like Excel and Video?"**
   - *Excel:* Transformed rows into natural language key-value sentences (`Column: Value`) alongside markdown tables so semantic dense embeddings can match numerical queries accurately.
   - *Video:* Sampled keyframes at fixed time intervals (every 2 seconds) with OpenCV, storing the timestamp `MM:SS` and image file path in the chunk metadata so retrieval pinpoints the exact second.
2. **"Why did you use local HuggingFace embeddings instead of an API?"**
   - Avoids API rate limits (HTTP 429/503), reduces operational cost to zero, and allows 100% offline or on-premise execution.
3. **"How does ChromaDB handle multimodal retrieval?"**
   - ChromaDB stores document vectors and metadata dictionaries (e.g., `modality='video_keyframe'`, `timestamp='00:02'`). On query match, the system extracts the metadata path and displays the image or frame inline.